# F02 OCULUS — PALATIUM
> *Frégate Viewer — Configuration Caméra, Éclairage, Tags*

**Pipeline :**
1. Monter Drive
2. Valider les fichiers IN/ avec L'Arbitre
3. Démarrer Flask + ngrok
4. Accéder au viewer via l'URL ngrok
5. Configurer spawn, waypoints, éclairage, tags
6. Sauvegarder (Ctrl+S dans le viewer)
7. Valider OUT/ avec L'Arbitre

---
**Doctrine :** F02 ne lit que `F02_OCULUS/IN/` et n'écrit que dans `F02_OCULUS/OUT/`.

In [ ]:
# ─── CONFIGURATION ─────────────────────────────────────────
DRIVE_ROOT  = '/content/drive/MyDrive/DRIVE_PALATIUM'
FLASK_PORT  = 5002
NGROK_TOKEN = ''  # ← Votre token ngrok (gratuit sur ngrok.com)
# ───────────────────────────────────────────────────────────

## Étape 1 — Monter Drive + Installer dépendances

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install flask flask-cors pyngrok -q
print('✅ Setup complet')

## Étape 2 — Valider les fichiers IN/ (L'Arbitre CHECK-IN)

In [ ]:
import shutil
shutil.copy(f'{DRIVE_ROOT}/PAL_ARBITRE.py', '/content/PAL_ARBITRE.py')

result = !python /content/PAL_ARBITRE.py \
    --frigate F02 \
    --mode check-in \
    --drive-root {DRIVE_ROOT}

for line in result: print(line)

# Vérifier le code de sortie
import subprocess
ret = subprocess.run(['python', '/content/PAL_ARBITRE.py',
    '--frigate', 'F02', '--mode', 'check-in', '--drive-root', DRIVE_ROOT],
    capture_output=True)
if ret.returncode != 0:
    raise RuntimeError('F02 BLOQUÉE — Transférer les fichiers IN/ avant de continuer')
print('✅ F02 IN/ validé — prêt au lancement')

## Étape 3 — Copier les scripts dans Colab

In [ ]:
import shutil, os

CODEBASE = f'{DRIVE_ROOT}/F02_OCULUS/CODEBASE'

for f in ['pal_f02_flask.py', 'pal_f02_viewer.html']:
    shutil.copy(f'{CODEBASE}/{f}', f'/content/{f}')
    print(f'✅ {f}')

## Étape 4 — Démarrer Flask + ngrok

In [ ]:
import threading
from pyngrok import ngrok, conf

# Configurer ngrok
if NGROK_TOKEN:
    conf.get_default().auth_token = NGROK_TOKEN

# Démarrer Flask dans un thread
import sys
sys.path.insert(0, '/content')

from pal_f02_flask import app, init_config
init_config(DRIVE_ROOT)

def run_flask():
    app.run(host='0.0.0.0', port=FLASK_PORT, debug=False, use_reloader=False)

thread = threading.Thread(target=run_flask, daemon=True)
thread.start()

import time; time.sleep(1)

# Tunnel ngrok
public_url = ngrok.connect(FLASK_PORT)
print(f'\n  ✅ VIEWER DISPONIBLE ICI :')
print(f'  🔗 {public_url}')
print(f'\n  Ouvrir ce lien dans votre navigateur.')
print(f'  Raccourcis : S = Spawn | W = Waypoint | Ctrl+S = Sauvegarder')

## Étape 5 — Utiliser le viewer

1. Ouvrir l'URL ngrok ci-dessus dans votre navigateur
2. Naviguer dans la scène 3D
3. Définir le spawn (`S`) et les waypoints (`W`)
4. Sélectionner le preset caméra et l'ambiance HDRI
5. Valider/corriger les tags (lampes, vitres, portes)
6. Appuyer `Ctrl+S` pour sauvegarder

Une fois terminé, exécuter la cellule ci-dessous.

## Étape 6 — Valider les fichiers OUT/ (L'Arbitre CHECK-OUT)

In [ ]:
import subprocess
ret = subprocess.run(['python', '/content/PAL_ARBITRE.py',
    '--frigate', 'F02', '--mode', 'check-out',
    '--drive-root', DRIVE_ROOT, '--verbose'],
    capture_output=False)

if ret.returncode == 0:
    print('\n  ✅ F02 OCULUS SCELLÉE — Transférer les configs vers F03')
else:
    print('\n  ❌ Configs manquantes — Sauvegarder depuis le viewer (Ctrl+S)')

---
## Résumé Transit

Après CHECK-OUT ✅ :

| Fichier | Source | Destination |
|---------|--------|-------------|
| `creative_config.json` | `F02_OCULUS/OUT/` | `F03_SCRIPTORIUM/IN/` |
| `tags_config.json` | `F02_OCULUS/OUT/` | `F03_SCRIPTORIUM/IN/` |

**Inscrire dans `TRACKING/PALATIUM_TRANSFER_LOG.md`.**

*F02 OCULUS — Scellée. Ad Victoriam.*